# Notebook 8: Per-label analysis of a trained backbone

Diagnostics on a fully trained model (Notebook 2 checkpoint `checkpoints/backbone_ccs.pt`, or the Notebook 6
`baseline`). The Hypothesis A and C experiments in the report were run on an early checkpoint; this notebook
recomputes the corresponding descriptive analyses on the converged model and adds three that the label
definition makes necessary.

1. **Frequency.** How much of the per-label AUPRC does training frequency explain?
2. **Recurrent vs new positives.** Notebook 1 keeps *all* codes of visit $k+1$ as targets, including codes
   already present in the history. A recurrent positive can be predicted by copying the history; a new one
   cannot. We measure the share of recurrent positives per frequency quartile and the AUPRC on new
   occurrences only.
3. **Contextual diversity** $\delta_r$ (Hypothesis D): distinct co-occurrence sets per occurrence, and its
   association with AUPRC at fixed frequency (partial Spearman).
4. **Embedding geometry.** Does the learned $Z_{dx}$ organise codes by clinical chapter, and is this weaker
   for rare codes? Clustering quality (silhouette, AMI with CCS chapters) and nearest-neighbour agreement by
   input-frequency quartile.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path(os.environ.get('RAREPRED_ROOT', '..')).resolve()
sys.path.insert(0, str(ROOT / 'src'))
PROCESSED   = Path(os.environ.get('RAREPRED_PROCESSED', ROOT / 'processed'))
EXTERNAL    = ROOT / 'data' / 'external'
RESULTS     = ROOT / 'results'
FIGURES     = ROOT / 'figures'
CHECKPOINTS = ROOT / 'checkpoints'
for d in (RESULTS, FIGURES, CHECKPOINTS):
    d.mkdir(parents=True, exist_ok=True)

import numpy as np, pandas as pd, torch, collections
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_mutual_info_score, average_precision_score
from rarepred.experiment import Setup
from rarepred.models import MultiModalBackbone
from rarepred.train import predict
from rarepred.metrics import per_label_ap
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
S = Setup(PROCESSED, DEVICE, knowledge_npz=PROCESSED / f"knowledge_embeddings_{os.environ.get('RAREPRED_ENCODER_TAG', 'BioLORD-2023')}.npz")
CKPT = Path(os.environ.get('RAREPRED_CKPT', CHECKPOINTS / 'backbone_ccs.pt'))
if not CKPT.exists():
    CKPT = CHECKPOINTS / 'baseline_seed42.pt'
state = torch.load(CKPT, map_location=DEVICE)
state = state.get('model', state)
model = MultiModalBackbone(*S.C, S.D_out).to(DEVICE)
model.load_state_dict(state)
print('checkpoint:', CKPT.name)
probs, ys = predict(model, S.test_loader, DEVICE)
ap = per_label_ap(probs, ys)
freq = S.freq_sample
grp = np.empty(S.D_out, dtype=object)
for g, idx in S.groups_sample.items():
    grp[idx] = g
QS = list(S.groups_sample)

import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 200, 'font.size': 9, 'axes.titlesize': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#666666', 'axes.labelcolor': '#191919', 'xtick.color': '#666666',
    'ytick.color': '#666666', 'axes.grid': True, 'grid.color': '#e6e6e6', 'grid.linewidth': 0.6,
    'axes.axisbelow': True, 'legend.frameon': False,
})
BLUE, ORANGE, AQUA, GREY = '#257de0', '#f47230', '#00b894', '#9c9c9c'

## 1. Frequency

In [ ]:
ok = np.isfinite(ap) & (freq > 0)
r, p = stats.pearsonr(np.log(freq[ok]), ap[ok])
rho, prho = stats.spearmanr(freq[ok], ap[ok])
print(f"labels with >= 1 test positive: {ok.sum()} of {S.D_out}")
print(f"AP ~ log(n): Pearson r = {r:.3f} (R^2 = {r*r:.3f}, p = {p:.1e}); Spearman rho = {rho:.3f}")
by_q = pd.DataFrame({'group': grp[ok], 'ap': ap[ok]}).groupby('group').ap.describe()[['count', 'mean', '25%', '50%', '75%']]
by_q.loc[QS].round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 3.6))
cols = dict(zip(QS, ['#0d366b', '#1c5cab', '#3987e5', '#86b6ef']))
for g in QS:
    m = ok & (grp == g)
    ax.scatter(freq[m], ap[m], s=9, color=cols[g], label=g, linewidths=0)
xx = np.linspace(np.log(freq[ok]).min(), np.log(freq[ok]).max(), 50)
b1, b0 = np.polyfit(np.log(freq[ok]), ap[ok], 1)
ax.plot(np.exp(xx), b0 + b1 * xx, color='#191919', lw=1.2)
ax.set_xscale('log'); ax.set_xlabel('training samples containing the label'); ax.set_ylabel('test AUPRC (per label)')
ax.legend(fontsize=7, markerscale=1.5); ax.set_title(f'Per-label AUPRC vs frequency (R$^2$ = {r*r:.2f})', loc='left')
plt.tight_layout(); plt.savefig(FIGURES / 'per_label_auprc_vs_frequency.png', bbox_inches='tight'); plt.show()

### 1b. Reconciling with MedCo's frequency groups

MedCo's evaluation code (`utils/eval_test.py`, `get_group_labels1`) sorts labels by frequency and assigns each
label to the group in which the *cumulative share of label occurrences* falls: each group holds a quarter of
the occurrences, not a quarter of the labels. Its "0-25%" group therefore contains most of the label
vocabulary. The cell below evaluates the same model under both definitions.

In [ ]:
from rarepred.metrics import group_sample_auprc
rows = []
for gname, groups in [('equal-size label quartiles (this report)', S.groups_sample),
                      ('cumulative-occurrence groups (MedCo code)', S.groups_mass)]:
    for g, idx in groups.items():
        val, n = group_sample_auprc(probs, ys, idx)
        rows.append({'definition': gname, 'group': g, 'labels': len(idx), 'test samples': n, 'sample-AUPRC': round(val, 4)})
medco_cmp = pd.DataFrame(rows).set_index(['definition', 'group'])
medco_cmp

## 2. Recurrent vs new positives

In [ ]:
def history_labels(samples):
    H = np.zeros((len(samples), S.D_out), dtype=bool)
    for i, s in enumerate(samples):
        for _, dx, _, _ in S.seqs[s['patient_id']][:s['k']]:
            for c in dx:
                j = S.code2idx.get(c)
                if j is not None:
                    H[i, j] = True
    return H

H_test = history_labels(S.test_samples)
Yb = ys.astype(bool)
rec_share = np.array([(Yb[:, j] & H_test[:, j]).sum() / max(1, Yb[:, j].sum()) for j in range(S.D_out)])
# AUPRC on new occurrences: drop, for label j, the test samples whose history already contains j
ap_new = np.full(S.D_out, np.nan)
for j in range(S.D_out):
    m = ~H_test[:, j]
    if Yb[m, j].sum() > 0:
        ap_new[j] = average_precision_score(Yb[m, j], probs[m, j])
tab = pd.DataFrame({'group': grp, 'recurrent share of positives': rec_share, 'AUPRC (all positives)': ap,
                    'AUPRC (new occurrences only)': ap_new})
recur = tab.groupby('group').mean(numeric_only=True).loc[QS].round(3)
recur

If the common quartile owes much of its AUPRC to recurrent codes, part of the rare/common gap measures how
often a label *recurs* rather than how hard it is to *anticipate*. The draft describes the target as new
diagnoses; with the current preprocessing the reported numbers are for all next-visit codes, and the
new-occurrence column is the one that matches the stated task.

## 3. Contextual diversity

In [ ]:
train_pids = {s['patient_id'] for s in S.train_samples}
occ, ctx = collections.Counter(), collections.defaultdict(set)
for pid in train_pids:
    for _, dx, _, _ in S.seqs[pid]:
        dx = set(dx)
        for c in dx:
            if c in S.code2idx:
                occ[c] += 1
                ctx[c].add(frozenset(dx - {c}))
delta = np.array([len(ctx[c]) / occ[c] if occ[c] else np.nan for c in S.vocab])
n_occ = np.array([occ[c] for c in S.vocab], dtype=float)
m = np.isfinite(delta) & np.isfinite(ap) & (n_occ > 0)
def resid(y, x):
    ry, rx = stats.rankdata(y), stats.rankdata(x)
    b = np.polyfit(rx, ry, 1)
    return ry - np.polyval(b, rx)
pr, pp = stats.pearsonr(resid(ap[m], np.log(n_occ[m])), resid(delta[m], np.log(n_occ[m])))
print(f"Spearman(delta, AP)            = {stats.spearmanr(delta[m], ap[m])[0]:.3f}")
print(f"partial Spearman(delta, AP | n) = {pr:.3f} (p = {pp:.1e}, {m.sum()} labels)")
print(f"Spearman(delta, n)             = {stats.spearmanr(delta[m], n_occ[m])[0]:.3f}")

$\delta_r$ is mechanically high for rare codes (few occurrences rarely repeat a context), so the raw
correlation with AUPRC is confounded by frequency; only the partial coefficient is informative.

## 4. Geometry of the learned diagnosis embeddings

In [ ]:
W = model.emb_dx.weight.detach().cpu().numpy()
codes = sorted(S.dx2idx, key=S.dx2idx.get)
Z = W[[S.dx2idx[c] for c in codes]]
Zn = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12)
chapter = np.array([c.split('.')[0] for c in codes])
in_freq = np.array([S.counts['dx'].get(c, 0) for c in codes], dtype=float)
rows = []
for k in [10, 18, 25, 40]:
    lab = KMeans(k, n_init=10, random_state=0).fit_predict(Zn)
    rows.append({'k': k, 'silhouette (cosine)': silhouette_score(Zn, lab, metric='cosine'),
                 'AMI with CCS chapter': adjusted_mutual_info_score(chapter, lab)})
clus = pd.DataFrame(rows).set_index('k').round(3)
display(clus)
Sim = Zn @ Zn.T; np.fill_diagonal(Sim, -np.inf)
nn5 = np.argsort(-Sim, axis=1)[:, :5]
agree = (chapter[nn5] == chapter[:, None]).mean(1)
qs_in = np.percentile(in_freq, [25, 50, 75])
qin = np.digitize(in_freq, qs_in, right=True)
nn_tab = pd.DataFrame({'input-frequency quartile': qin + 1, 'agree': agree}).groupby('input-frequency quartile').agree.agg(['count', 'mean']).round(3)
print('5-NN agreement with CCS chapter (chance level is about', round(float(pd.Series(chapter).value_counts(normalize=True).pow(2).sum()), 3), ')')
nn_tab

In [ ]:
if S.knowledge is not None:
    kz = dict(zip(list(S.knowledge['dx_codes']), S.knowledge['dx_rag']))
    have = [i for i, c in enumerate(codes) if c in kz]
    T = np.stack([kz[codes[i]] for i in have]); T /= np.linalg.norm(T, axis=1, keepdims=True)
    St = T @ T.T; np.fill_diagonal(St, -np.inf)
    Sz = Sim[np.ix_(have, have)]
    k = 10
    a, b = np.argsort(-Sz, 1)[:, :k], np.argsort(-St, 1)[:, :k]
    jac = np.array([len(set(a[i]) & set(b[i])) / len(set(a[i]) | set(b[i])) for i in range(len(have))])
    print(pd.DataFrame({'q': qin[have] + 1, 'jaccard@10': jac}).groupby('q').mean().round(3))
else:
    print('knowledge embeddings not found: alignment with text embeddings skipped')

## 5. Save

In [ ]:
out = {'checkpoint': CKPT.name, 'n_labels_evaluable': int(ok.sum()),
       'freq_regression': {'pearson_r': float(r), 'r2': float(r * r), 'p': float(p), 'spearman': float(rho)},
       'ap_by_quartile': by_q.loc[QS].reset_index().to_dict(orient='records'),
       'medco_group_comparison': medco_cmp.reset_index().to_dict(orient='records'),
       'recurrence': recur.reset_index().to_dict(orient='records'),
       'diversity': {'partial_spearman': float(pr), 'p': float(pp)},
       'clustering': clus.reset_index().to_dict(orient='records'),
       'nn_chapter_agreement': nn_tab.reset_index().to_dict(orient='records')}
with open(RESULTS / 'rare_code_analysis.json', 'w') as f:
    json.dump(out, f, indent=2)
print('saved results/rare_code_analysis.json')